In [2]:
import pandas as pd
import numpy as np

data = pd.read_csv('/content/drive/MyDrive/house_price_data.csv')


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/house_price_data.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data.sort_values('price_lakhs',ascending=False,inplace=True)


In [ ]:
data.corr()

In [ ]:

# import pandas as pd
# import numpy as np

# data = pd.read_csv('house_price_data.csv')

# print(data.shape)
# print(data.head())

# Independent variables
X = data.drop('price_lakhs', axis=1)

# Target  Dependent variable
y = data['price_lakhs']

print("Features:", X.columns.tolist())
print("Target:", y.name)


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


In [ ]:

from sklearn.tree import DecisionTreeRegressor

# Create model
dt_model = DecisionTreeRegressor(max_depth =8, min_samples_leaf= 2)


# Train model
dt_model.fit(X_train, y_train)

# Predictions
dt_pred = dt_model.predict(X_test)

print("Decision Tree Predictions:")
print(dt_pred[:10])


In [ ]:

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(22, 10))

plot_tree(
    dt_model,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=8
)

plt.show()


In [ ]:

from sklearn.ensemble import RandomForestRegressor

# Create model
rf_model = RandomForestRegressor(
    n_estimators=110,
    criterion='squared_error',
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=4,

)

# Train model
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)

print("Random Forest Predictions:")
print(rf_pred[:10])


In [ ]:

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

models = {
    'Decision Tree': dt_pred,
    'Random Forest': rf_pred
}

results = []

for name, predictions in models.items():

    results.append({
        'Model': name,
        'R2 Score': r2_score(y_test, predictions),
        'RMSE': np.sqrt(
            mean_squared_error(y_test, predictions)
        ),
        'MAE': mean_absolute_error(
            y_test, predictions
        )
    })

results_df = pd.DataFrame(results)

print(results_df.round(3))


In [15]:

# Decision Tree feature importance
dt_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
})

# Random Forest feature importance
rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

print("Decision Tree Importance")
print(dt_importance.sort_values(
    by='Importance', ascending=False
))

print("\nRandom Forest Importance")
print(rf_importance.sort_values(
    by='Importance', ascending=False
))



Decision Tree Importance
               Feature  Importance
0            area_sqft    0.844129
4  distance_to_city_km    0.070981
3   property_age_years    0.026791
5        school_rating    0.021019
1             bedrooms    0.014946
7           near_metro    0.010236
6       parking_spaces    0.006304
2            bathrooms    0.005594

Random Forest Importance
               Feature  Importance
0            area_sqft    0.852059
4  distance_to_city_km    0.066884
3   property_age_years    0.026542
5        school_rating    0.019725
1             bedrooms    0.015262
7           near_metro    0.008233
6       parking_spaces    0.006485
2            bathrooms    0.004811


In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

param_grid = {
    'n_estimators': [50, 70, 100, 200],
    'max_depth': [3, 4, 5, 8, 12, None],
    'min_samples_split': [2, 5, 10, 12],
    'min_samples_leaf': [1, 2, 3, 4]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("Best CV R2:")
print(grid.best_score_)

best_rf = grid.best_estimator_

best_pred = best_rf.predict(X_test)

print("Test R2:", r2_score(y_test, best_pred))

NameError: name 'X_train' is not defined

In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-optimize]


In [ ]:
from skopt import BayesSearchCV
from skopt.space import Categorical, Integer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Define the search space for Bayesian Optimization
# Note: max_depth is treated categorically to handle 'None' as an option
search_space = {
    'n_estimators': Integer(50, 200),
    'max_depth': Categorical([3, 4, 5, 8, 12, None]),
    'min_samples_split': Integer(2, 12),
    'min_samples_leaf': Integer(1, 4)
}

# Initialize BayesSearchCV
bayes_search = BayesSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    search_spaces=search_space,
    n_iter=32,          # Number of parameter settings that are sampled
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

# Run the search
bayes_search.fit(X_train, y_train)

print("Best Parameters Found by Bayesian Search:")
print(bayes_search.best_params_)

print("\nBest CV R2 Score:")
print(bayes_search.best_score_)

# Evaluate on the test set
best_bayes_rf = bayes_search.best_estimator_
bayes_pred = best_bayes_rf.predict(X_test)
print("\nTest R2 Score:", r2_score(y_test, bayes_pred))

Best Parameters Found by Bayesian Search:
OrderedDict({'max_depth': 12, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200})

Best CV R2 Score:
0.8745485916554421

Test R2 Score: 0.878110392519089
